In [ ]:
!pip -q install yfinance optuna scikit-learn joblib pandas numpy

In [ ]:
TICKER = "NVDA"                    # e.g., AAPL, MSFT, GOOGL, META, AMZN, TSLA, NVDA
START  = "2015-01-01"
END    = "2025-08-01"
STUDENT_ID = "311438"
SEED_INDEX = 0
N_SPLITS   = 5
N_TRIALS   = 30
METRIC     = "roc_auc"
OUTPUT_DIR = "/home/nvida/"

print("OK: config loaded.")


OK: config loaded.


In [ ]:
import hashlib, os, random, json
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
import optuna
from optuna.samplers import TPESampler

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix
)
import joblib

def derive_seeds_from_student_id(student_id: str, n_seeds: int = 5):
    base = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    seeds = []
    for i in range(n_seeds):
        chunk = base[i*8:(i+1)*8]  # 8 hex chars -> 32-bit int
        if len(chunk) < 8:
            chunk = (chunk + base)[:8]
        seeds.append(int(chunk, 16))
    return seeds

def download_close_series(ticker: str, start: str, end: str, interval: str = "1d") -> pd.DataFrame:
    data = yf.download(ticker, start=start, end=end, interval=interval, auto_adjust=False, progress=False)
    if data.empty or "Close" not in data.columns:
        raise ValueError(f"No data/Close column returned for {ticker}.")
    df = data[["Close"]].copy()
    df.reset_index(inplace=True)
    # yfinance sometimes returns "Date" or "Datetime"
    if "Date" not in df.columns and "Datetime" in df.columns:
        df.rename(columns={"Datetime": "Date"}, inplace=True)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    return df

def derive_labels_from_close(df: pd.DataFrame, threshold_quantile: float = 0.75):
    df = df.copy()
    df["log_ret"] = np.log(df["Close"]).diff()
    df = df.iloc[1:].reset_index(drop=True)  # drop first NaN
    abs_ret = df["log_ret"].abs().values
    thresh = np.quantile(abs_ret[~np.isnan(abs_ret)], threshold_quantile)
    y = (abs_ret >= thresh).astype(int)
    return df, y  # df has Date, Close, log_ret

def build_close_only_features(df: pd.DataFrame) -> pd.DataFrame:
    feats = pd.DataFrame(index=df.index)
    feats["ret1"]  = df["log_ret"]                 # 1-day log return (already computed)
    feats["ret5"]  = np.log(df["Close"]).diff(5)   # 5-day log return
    feats["ma5"]   = df["Close"].rolling(5).mean()
    feats["ma10"]  = df["Close"].rolling(10).mean()
    feats["std5"]  = df["Close"].rolling(5).std()
    feats["std10"] = df["Close"].rolling(10).std()
    feats = feats.replace([np.inf, -np.inf], np.nan).dropna().copy()
    return feats

def run_training(ticker, start, end, student_id, seed_index, n_splits, n_trials, metric, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # 1) Data
    raw = download_close_series(ticker, start, end, interval="1d")

    # 2) Labels from closing price
    df2, y_full = derive_labels_from_close(raw, threshold_quantile=0.75)

    # 3) Features (closing-only)
    feats = build_close_only_features(df2)

    # Align y to features index and clean
    y = pd.Series(y_full, index=df2.index).reindex(feats.index).values
    mask = ~np.isnan(y)
    X = feats.values[mask]
    y = y[mask].astype(int)

    # 4) Scale
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xs = scaler.fit_transform(X)

    # 5) Seeds — derive 5, use only ONE
    seeds = derive_seeds_from_student_id(student_id, n_seeds=5)
    if not (0 <= seed_index < len(seeds)):
        raise ValueError(f"seed_index must be 0..4; got {seed_index}")
    seed = seeds[seed_index]
    np.random.seed(seed); random.seed(seed)

    print("Derived seeds from student ID:", seeds)
    print(f"Using ONLY seed index {seed_index}: {seed}")

    # 6) CV + Optuna tuning
    tscv = TimeSeriesSplit(n_splits=n_splits)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=100),
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "max_features": trial.suggest_categorical("max_features", ["sqrt","log2", None]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
            "random_state": seed,
            "n_jobs": -1,
        }
        clf = RandomForestClassifier(**params)

        scores = []
        for tr, va in tscv.split(Xs):
            Xtr, Xva = Xs[tr], Xs[va]
            ytr, yva = y[tr], y[va]
            clf.fit(Xtr, ytr)
            prob = clf.predict_proba(Xva)[:,1]
            if metric == "roc_auc":
                score = roc_auc_score(yva, prob)
            elif metric == "average_precision":
                score = average_precision_score(yva, prob)
            else:
                pred = (prob >= 0.5).astype(int)
                score = f1_score(yva, pred)
            scores.append(score)
        return float(np.mean(scores))

    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params = study.best_params
    best_params.update({"random_state": seed, "n_jobs": -1})
    best_model = RandomForestClassifier(**best_params).fit(Xs, y)

    # 7) Simple holdout: last fold
    last_tr, last_va = list(TimeSeriesSplit(n_splits=n_splits).split(Xs))[-1]
    Xtr, Xho = Xs[last_tr], Xs[last_va]
    ytr, yho = y[last_tr], y[last_va]
    prob = best_model.predict_proba(Xho)[:,1]
    pred = (prob >= 0.5).astype(int)

    metrics = {
        "ticker": ticker,
        "seed_used": seed,
        "seed_index": seed_index,
        "derived_seeds": seeds,
        "cv_metric": metric,
        "study_best_value": float(study.best_value),
        "holdout_roc_auc": float(roc_auc_score(yho, prob)),
        "holdout_average_precision": float(average_precision_score(yho, prob)),
        "holdout_f1": float(f1_score(yho, pred)),
        "confusion_matrix": confusion_matrix(yho, pred).tolist(),
        "classification_report": classification_report(yho, pred, output_dict=True),
        "used_feature_columns": list(feats.columns),
        "target_source": "derived_from_close_abs_log_returns_top_25pct",
        "target_name": "is_high_vol",
        "n_samples_total": int(len(y)),
        "n_samples_holdout": int(len(yho)),
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "optuna_best_params": best_params,
    }

    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(best_model, os.path.join(output_dir, "rf_optuna_model.joblib"))
    joblib.dump(scaler,      os.path.join(output_dir, "scaler.joblib"))
    with open(os.path.join(output_dir, "metrics.json"), "w") as f: json.dump(metrics, f, indent=2)
    with open(os.path.join(output_dir, "best_params.json"), "w") as f: json.dump(best_params, f, indent=2)

    print("\n=== SUMMARY ===")
    print(f"Ticker: {ticker}")
    print("Best CV ({}): {:.4f}".format(metric, study.best_value))
    print("Holdout ROC-AUC: {:.4f}".format(metrics["holdout_roc_auc"]))
    print("Holdout AP: {:.4f}".format(metrics["holdout_average_precision"]))
    print("Holdout F1: {:.4f}".format(metrics["holdout_f1"]))
    print("Features:", list(feats.columns))
    print("Target:", metrics["target_source"], f"({metrics['target_name']})")
    print("Artifacts saved to:", output_dir)

    return metrics

print("Functions ready.")

Functions ready.


SPlit based on mOnthly
train with one, two months
predict with 3rd month

Data needed to splited .


In [ ]:
metrics = run_training(
    ticker=TICKER,
    start=START,
    end=END,
    student_id=STUDENT_ID,
    seed_index=SEED_INDEX,
    n_splits=N_SPLITS,
    n_trials=N_TRIALS,
    metric=METRIC,
    output_dir=OUTPUT_DIR
)

[I 2025-08-29 00:37:08,687] A new study created in memory with name: no-name-3c5fec8b-44c0-44d9-8b0a-efc15e4d9a17


Derived seeds from student ID: [569498789, 554195645, 1795592910, 2769247983, 865482700]
Using ONLY seed index 0: 569498789


[I 2025-08-29 00:37:21,608] Trial 0 finished with value: 0.9983333333333334 and parameters: {'n_estimators': 700, 'max_depth': 19, 'min_samples_split': 19, 'min_samples_leaf': 13, 'max_features': 'log2', 'bootstrap': False, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9983333333333334.
[I 2025-08-29 00:37:43,486] Trial 1 finished with value: 0.9931982296762172 and parameters: {'n_estimators': 1000, 'max_depth': 16, 'min_samples_split': 13, 'min_samples_leaf': 10, 'max_features': None, 'bootstrap': False, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9983333333333334.
[I 2025-08-29 00:38:00,988] Trial 2 finished with value: 0.9995884773662551 and parameters: {'n_estimators': 1000, 'max_depth': 7, 'min_samples_split': 16, 'min_samples_leaf': 14, 'max_features': 'log2', 'bootstrap': True, 'class_weight': 'balanced'}. Best is trial 2 with value: 0.9995884773662551.
[I 2025-08-29 00:38:15,245] Trial 3 finished with value: 0.9931982296762172 and parameters: {'n_est


=== SUMMARY ===
Ticker: NVDA
Best CV (roc_auc): 0.9999
Holdout ROC-AUC: 1.0000
Holdout AP: 1.0000
Holdout F1: 1.0000
Features: ['ret1', 'ret5', 'ma5', 'ma10', 'std5', 'std10']
Target: derived_from_close_abs_log_returns_top_25pct (is_high_vol)
Artifacts saved to: /home/nvida/


In [ ]:
import json, os
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "r") as f:
    m = json.load(f)
m  # pretty-printed in Colab

{'ticker': 'NVDA',
 'seed_used': 569498789,
 'seed_index': 0,
 'derived_seeds': [569498789, 554195645, 1795592910, 2769247983, 865482700],
 'cv_metric': 'roc_auc',
 'study_best_value': 0.9998765432098764,
 'holdout_roc_auc': 1.0,
 'holdout_average_precision': 1.0,
 'holdout_f1': 1.0,
 'confusion_matrix': [[306, 0], [0, 135]],
 'classification_report': {'0': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 306.0},
  '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 135.0},
  'accuracy': 1.0,
  'macro avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 441.0},
  'weighted avg': {'precision': 1.0,
   'recall': 1.0,
   'f1-score': 1.0,
   'support': 441.0}},
 'used_feature_columns': ['ret1', 'ret5', 'ma5', 'ma10', 'std5', 'std10'],
 'target_source': 'derived_from_close_abs_log_returns_top_25pct',
 'target_name': 'is_high_vol',
 'n_samples_total': 2650,
 'n_samples_holdout': 441,
 'timestamp': '2025-08-29T00:46:07.316046Z',
 'o

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/M7_volatility"
!cp -r "{OUTPUT_DIR}" "/content/drive/MyDrive/M7_volatility/"
print("Copied to /content/drive/MyDrive/M7_volatility")

Mounted at /content/drive
Copied to /content/drive/MyDrive/M7_volatility


In [ ]:
from google.colab import files
files.download(f"{OUTPUT_DIR}/metrics.json")
files.download(f"{OUTPUT_DIR}/best_params.json")
files.download(f"{OUTPUT_DIR}/rf_optuna_model.joblib")
files.download(f"{OUTPUT_DIR}/scaler.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>